# Extended Case 4 — Perfect Parachute

R Jupyter version of the course material. Select an **R** kernel in Positron and run cells from top to bottom, starting with the setup cell. Keep the complete course folder, data and images together.

Generated from the matching QMD; make teaching edits in the QMD and regenerate. The published HTML/PDF retains the full course styling.


In [ ]:
# Notebook sessions may start in the notebook folder.
# Find the opened course folder so the original data/... paths work unchanged.
.course_root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(.course_root, "_quarto.yml")) ||
       !dir.exists(file.path(.course_root, "data"))) {
  .course_parent <- dirname(.course_root)
  if (identical(.course_parent, .course_root)) {
    stop("Open the complete course folder containing _quarto.yml and data/.")
  }
  .course_root <- .course_parent
}
setwd(.course_root)
rm(.course_root)
if (exists(".course_parent", inherits = FALSE)) rm(.course_parent)


In [ ]:
#| label: case-import
source("lessons/_includes/course-tools.R")
parachutes <- read.csv("data/business/extended-parachute.csv")
parachutes$supplier <- factor(parachutes$supplier)
stopifnot(nrow(parachutes) == 20, all(table(parachutes$supplier) == 5))
supplier_summary <- parachutes %>% group_by(supplier) %>%
  summarise(n = n(), mean_strength = mean(strength), sd_strength = sd(strength),
    min_strength = min(strength), max_strength = max(strength), .groups = "drop")
knitr::kable(supplier_summary, digits = 2)


- `data/business/extended-parachute.csv`; guide: `data/business/README.md`.
- Workbook means, medians and standard deviations are summary rows; they are excluded from the 20 observations.
- The means are 19.52, 24.26, 22.84 and 21.16 for suppliers 1–4.


In [ ]:
ggplot(parachutes, aes(supplier, strength)) +
  geom_boxplot(fill = petra_accent, width = .45, outlier.shape = NA) +
  geom_point(position = position_jitter(width = .06, height = 0, seed = 307312)) +
  labs(x = "Fibre supplier", y = "Tensile strength (source scale)")


- Individual points keep the five observations per supplier visible.
- A higher sample mean does not by itself establish a superior supplier.

## 3. A short ANOVA introduction

- Separate variation **between supplier means** from variation **within suppliers**.
- ANOVA divides each sum of squares by its degrees of freedom to form a mean square.
- **F = between-group mean square / within-group mean square.**
- A large F is evidence against equal population means under the model.
- **Null:** all four population mean strengths are equal.
- **Alternative:** at least one differs; the test does not say that all pairs differ.
- Classical one-way ANOVA assumes independent errors, approximately normal errors within groups and a common error variance.


In [ ]:
fit <- aov(strength ~ supplier, data = parachutes)
summary(fit)


- `aov()` fits the one-factor model; supplier is categorical, not a numeric trend.
- F is approximately **3.462**, with p approximately **0.0414**.
- At 5%, the model gives evidence against all four means being equal.

## 4. Which comparisons are supported?


In [ ]:
TukeyHSD(fit)


- Tukey’s procedure adjusts pairwise intervals and tests for the family of comparisons under the ANOVA assumptions.
- Read each contrast’s direction and interval; do not rank suppliers using p-values alone.
- An overall significant test need not make every pair significant.
- These intervals concern mean differences, not minimum guaranteed product strength.

## 5. Check the model and its limits


In [ ]:
par(mfrow = c(1, 2))
plot(fit, which = 1)
plot(fit, which = 2)
par(mfrow = c(1, 1))
fligner.test(strength ~ supplier, data = parachutes)


- Residual plots assess spread and approximate normality; they cannot establish independent assignment.
- `fligner.test()` checks dispersion; a large p-value does not prove equal variances.
- Five observations per supplier give imprecise diagnostics.
- Causal supplier claims need a defensible assignment protocol and control of weaving/testing conditions.

### Optional: relax the equal-variance assumption


In [ ]:
oneway.test(strength ~ supplier, data = parachutes, var.equal = FALSE)


- Welch’s one-way test avoids assuming equal variances.
- Tukey intervals from the classical model do not automatically become Welch-compatible pairwise intervals.
- Report sensitivity openly rather than choosing the method giving the preferred answer.

## 6. Independent procurement briefing

1.  Explain F using the observed between- and within-supplier variation.
2.  Identify the largest sample mean difference and report its Tukey interval and adjusted p-value.
3.  Compare classical and Welch conclusions, including their assumptions.
4.  Explain why five tests per supplier cannot establish rare failure rates or minimum guaranteed strength.
5.  Specify additional evidence on cost, delivery reliability and engineering requirements before recommending a supplier.
6.  Propose a follow-up experiment controlling loom conditions and assigning fibre treatments appropriately.

## 7. Optional design discussion: supplier and loom

- The book also considers four suppliers and two loom types.
- The local `Parachute2` workbook contains a separate `Data` sheet with loom-level observations as well as summary sheets.
- This case analyses the original **one-way, 20-measurement dataset**; it does not combine the two experiments.
- A supplier-by-loom interaction means the supplier difference depends on loom type.
- Discuss how an interaction would change procurement advice; a two-factor analysis is beyond this assignment.

## Submission

- Supplier summary table, one chart and a brief explanation of the overall and pairwise results.
- A 150-word procurement memo with a practical threshold, assumptions and a proposed follow-up.
- If a minimum strength threshold is not supplied, request one; do not invent an engineering requirement.
